In [1]:

#@title 🔧 Imports, seeds, device
import os, random, time
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import torchvision
from torchvision import transforms

from tqdm import tqdm

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [2]:
#@title ⚙️ PM-SFDA Configuration (Conference Version – Tuned)

import os

class CFG:
    # =====================================================
    # Dataset
    # =====================================================
    source_name = "WHU-RS19"
    target_name = "EuroSAT"
    num_classes = 6

    # =====================================================
    # Source Training (Supervised)
    # =====================================================
    src_epochs = 3            # ⬆️ 5 is too small → weak source model
    src_batch_size = 64
    src_lr = 1e-3

    # =====================================================
    # PM-SFDA Adaptation (Source-Free)
    # =====================================================
    sf_epochs = 3               # ⬆️ 7 epochs is insufficient for SFDA
    sf_batch_size = 64
    sf_lr =1e-4 #1e-5                 # ⬇️ Safe LR for ResNet-18 SFDA

    # =====================================================
    # Pseudo-Labeling (CRITICAL)
    # =====================================================
    
    conf_start = 0.90            # ⬇️ Less strict start → more samples
    conf_end = 0.70              # ⬇️ Necessary for WHU-RS19
    
    # =====================================================
    # Loss Weights (Conference Version)
    # =====================================================
    lambda_pl = 1.0              # Main supervision
    lambda_proto = 0.8#0.7           # Strong but not dominant
    lambda_cons = 0.2#0.3            # ⬇️ Consistency should not dominate

    # =====================================================
    # Consistency
    # =====================================================
    temp_cons = 0.5              # Stable for KL consistency

    # =====================================================
    # Model / Features
    # =====================================================
    feat_dim = 512               # Native ResNet-18 features
    
    # =====================================================
    # Misc
    # =====================================================
    num_workers = 0
    out_dir = "./pm_sfda_ckpts_whu_to_eurosat"


# Create output directory
os.makedirs(CFG.out_dir, exist_ok=True)

print("✅ PM-SFDA configuration tuned for GOOD accuracy")


✅ PM-SFDA configuration tuned for GOOD accuracy


In [3]:
#@title 📦 PM-SFDA Data Pipeline (EuroSAT → WHU-RS19) — FIXED

from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
import os

# =====================================================
# Normalization (ImageNet – correct for ResNet-18)
# =====================================================
mean_rgb = (0.485, 0.456, 0.406)
std_rgb  = (0.229, 0.224, 0.225)

# =====================================================
# TARGET DOMAIN (WHU-RS19)
# =====================================================

# Weak view → pseudo-label generation
tgt_weak_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean_rgb, std_rgb),
])

# Strong view → consistency regularization
tgt_strong_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(256, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.15, contrast=0.15,
        saturation=0.15, hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean_rgb, std_rgb),
])

# =====================================================
# SOURCE DOMAIN (EuroSAT)
# =====================================================

src_train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean_rgb, std_rgb),
])

src_test_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean_rgb, std_rgb),
])

# =====================================================
# Dataset Loading (FIXED PATHS)
# =====================================================

# ✅ Source datasets (whu)
src_train = datasets.ImageFolder(
    root="./data/source_whu_split/train",
    transform=src_train_tf
)

src_test = datasets.ImageFolder(
    root="./data/source_whu_split/test",   # ✅ FIXED
    transform=src_test_tf
)

# ✅ Target datasets (EuroSAT)
tgt_train_raw = datasets.ImageFolder(
    root="./data/target_eurosat_split/train",
    transform=None
)

tgt_test = datasets.ImageFolder(
    root="./data/target_eurosat_split/test",
    transform=tgt_weak_tf
)

print(f"Source (WHU-RS19 ): train={len(src_train)}, test={len(src_test)}")
print(f"Target (EuroSAT): train={len(tgt_train_raw)}, test={len(tgt_test)}")

# =====================================================
# Dual-view dataset for PM-SFDA
# =====================================================

class TwoViewRemoteSensing(Dataset):
    """
    Returns:
        x_w : weakly augmented image (Tensor)
        x_s : strongly augmented image (Tensor)
        -1  : dummy label (target is unlabeled)
    """
    def __init__(self, base_dataset, weak_tf, strong_tf):
        self.base = base_dataset
        self.weak_tf = weak_tf
        self.strong_tf = strong_tf

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _ = self.base[idx]   # PIL image
        xw = self.weak_tf(img)    # Tensor
        xs = self.strong_tf(img)  # Tensor
        return xw, xs, -1

# =====================================================
# DataLoaders (FIXED)
# =====================================================

# Source loaders
src_train_loader = DataLoader(
    src_train,
    batch_size=CFG.src_batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=True
)

src_test_loader = DataLoader(
    src_test,
    batch_size=CFG.src_batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True
)

# Target loaders
tgt_train = TwoViewRemoteSensing(
    tgt_train_raw,
    tgt_weak_tf,
    tgt_strong_tf
)

tgt_unl_loader = DataLoader(
    tgt_train,                 # ✅ FIXED
    batch_size=CFG.sf_batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=True
)

tgt_test_loader = DataLoader(
    tgt_test,
    batch_size=CFG.sf_batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True
)


Source (WHU-RS19 ): train=227, test=100
Target (EuroSAT): train=11200, test=4800


In [4]:
#@title 🧠 PM-SFDA Model (ResNet-18 for RGB Images)

import torch
import torch.nn as nn
from torchvision import models

class ResNetFeatureExtractor(nn.Module):
    """
    ResNet-18 backbone for PM-SFDA (Conference Version)

    Design choices:
    - Native 512-D feature space (no projection head)
    - Simple, interpretable prototype memory
    - Classifier can be frozen during SFDA
    """

    def __init__(self, num_classes=6, in_channels=3, use_pretrained=True):
        super().__init__()

        # -------------------------------
        # Backbone: ResNet-18
        # -------------------------------
        self.backbone = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1
            if use_pretrained else None
        )

        # Adapt first conv if input channels ≠ 3 (kept for completeness)
        if in_channels != 3:
            self.backbone.conv1 = nn.Conv2d(
                in_channels, 64,
                kernel_size=7, stride=2, padding=3, bias=False
            )

        # Feature extractor up to global average pooling
        self.features = nn.Sequential(
            self.backbone.conv1,
            self.backbone.bn1,
            self.backbone.relu,
            self.backbone.maxpool,
            self.backbone.layer1,
            self.backbone.layer2,
            self.backbone.layer3,
            self.backbone.layer4,
            self.backbone.avgpool     # [B, 512, 1, 1]
        )

        # Feature dimension (fixed for ResNet-18)
        self.feat_dim = 512

        # -------------------------------
        # Classifier head
        # -------------------------------
        self.classifier = nn.Linear(self.feat_dim, num_classes)

        # Initialization (stable for SFDA)
        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.classifier.bias, 0.0)

    def forward(self, x, return_feat=False):
        # Backbone feature extraction
        f = self.features(x)
        f = f.view(f.size(0), -1)   # [B, 512]

        logits = self.classifier(f)

        if return_feat:
            # Return logits + features for prototype & consistency
            return logits, f
        return logits


In [5]:
def accuracy(model, loader):
    """Computes classification accuracy."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    model.train()
    return 100.0 * correct / max(1, total)


In [6]:
# Initialize model
src_model = ResNetFeatureExtractor(
    num_classes=CFG.num_classes,
    in_channels=3,
    use_pretrained=True
).to(device)

print(src_model)
print("✅ PM-SFDA ResNet-18 model initialized (512-D features)")


ResNetFeatureExtractor(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine

In [7]:
#@title 🏋️ Train source model on EuroSAT (Supervised, PM-SFDA)

import torch
import torch.nn as nn
from tqdm import tqdm

def train_source(
    model,
    train_loader,
    test_loader,
    epochs=CFG.src_epochs,
    lr=CFG.src_lr
):
    """
    Supervised source training for PM-SFDA.
    - Freezes early layers for speed & stability
    - Uses AMP when CUDA is available
    - Produces a clean source model (before SFDA)
    """

    # -------------------------------------------------
    # Freeze early layers (recommended for EuroSAT)
    # -------------------------------------------------
    for name, param in model.named_parameters():
        if any(k in name for k in ["conv1", "layer1", "layer2"]):
            param.requires_grad = False

    # -------------------------------------------------
    # Optimizer (ONLY trainable parameters)
    # -------------------------------------------------
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss()

    # -------------------------------------------------
    # Mixed Precision (safe)
    # -------------------------------------------------
    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best = -1.0

    # -------------------------------------------------
    # Training Loop
    # -------------------------------------------------
    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        pbar = tqdm(train_loader, desc=f"[Source] Epoch {ep}/{epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        # -------------------------------------------------
        # Validation
        # -------------------------------------------------
        acc_src = accuracy(model, test_loader)
        best = max(best, acc_src)

        print(
            f"[Source] Epoch {ep:03d} | "
            f"Loss={running_loss/len(train_loader):.4f} | "
            f"ValAcc={acc_src:.2f}% (Best {best:.2f}%)"
        )

    return model
# Initialize PM-SFDA source model
src_model = ResNetFeatureExtractor(
    num_classes=CFG.num_classes,
    in_channels=3
).to(device)

print(f"Model initialized: {type(src_model).__name__}")
# Train source model
src_model = train_source(
    src_model,
    src_train_loader,
    src_test_loader,
    epochs=CFG.src_epochs,
    lr=CFG.src_lr
)
src_ckpt_path = os.path.join(CFG.out_dir, "source_model.pth")
torch.save(src_model.state_dict(), src_ckpt_path)
print("✅ Saved source model to:", src_ckpt_path)
src_only_acc_on_target = accuracy(src_model, tgt_test_loader)
print(f"📊 Source-only accuracy on target test: {src_only_acc_on_target:.2f}%")


Model initialized: ResNetFeatureExtractor


[Source] Epoch 1/3: 100%|██████████| 3/3 [00:11<00:00,  3.74s/it, loss=0.5667]


[Source] Epoch 001 | Loss=1.1485 | ValAcc=85.00% (Best 85.00%)


[Source] Epoch 2/3: 100%|██████████| 3/3 [00:02<00:00,  1.37it/s, loss=0.1243]


[Source] Epoch 002 | Loss=0.1942 | ValAcc=95.00% (Best 95.00%)


[Source] Epoch 3/3: 100%|██████████| 3/3 [00:02<00:00,  1.37it/s, loss=0.0475]


[Source] Epoch 003 | Loss=0.0605 | ValAcc=96.00% (Best 96.00%)
✅ Saved source model to: ./pm_sfda_ckpts_whu_to_eurosat\source_model.pth
📊 Source-only accuracy on target test: 39.94%


In [8]:
src_ckpt_path = os.path.join(CFG.out_dir, "source_model.pth")
torch.save(src_model.state_dict(), src_ckpt_path)
print("✅ Saved source model to:", src_ckpt_path)

✅ Saved source model to: ./pm_sfda_ckpts_whu_to_eurosat\source_model.pth


In [9]:
xw, xs, _ = next(iter(tgt_unl_loader))
print(type(xw), xw.shape)
print(type(xs), xs.shape)


<class 'torch.Tensor'> torch.Size([64, 3, 256, 256])
<class 'torch.Tensor'> torch.Size([64, 3, 256, 256])


In [11]:
#@title 🔁 PM-SFDA Adaptation (Prototype + Consistency, Conference) — UPDATED (No collapse)

import torch
import torch.nn as nn
import torch.nn.functional as F
import os
from tqdm import tqdm

# =====================================================
# Helper Functions
# =====================================================

def freeze_bn(m):
    """Freeze BatchNorm running stats (source-free stability)."""
    if isinstance(m, nn.BatchNorm2d):
        m.eval()

def conf_thresh(epoch, max_epoch, start=CFG.conf_start, end=CFG.conf_end, floor=0.40):
    """
    Linear schedule with a LOW floor for small WHU train sets.
    If floor is too high (0.70), you may accept zero samples.
    """
    th = start - (start - end) * (epoch / max_epoch)
    return max(th, floor)

def consistency_kl(logits_w, logits_s, T=CFG.temp_cons):
    """Weak→Strong KL (temperature-scaled)."""
    p_w = torch.softmax(logits_w / T, dim=1).detach()
    log_p_s = torch.log_softmax(logits_s / T, dim=1)
    return F.kl_div(log_p_s, p_w, reduction="batchmean") * (T * T)

def proto_loss(feats, labels, protos):
    """Cosine prototype attraction: 1 - cos(feat, proto[label])."""
    feats = F.normalize(feats, dim=1)
    return (1.0 - (feats * protos[labels]).sum(dim=1)).mean()

@torch.no_grad()
def confidence_stats(model, loader, max_batches=None):
    """Debug: show confidence distribution on target."""
    model.eval()
    confs = []
    for bi, (xw, _, _) in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        xw = xw.to(device)
        logits, _ = model(xw, return_feat=True)
        conf = torch.softmax(logits, dim=1).max(dim=1)[0]
        confs.append(conf.detach().cpu())
    if len(confs) == 0:
        return None
    confs = torch.cat(confs)
    return float(confs.min()), float(confs.mean()), float(confs.max())

# =====================================================
# Load Source Model
# =====================================================

src_ckpt_path = os.path.join(CFG.out_dir, "source_model.pth")
print("📁 Loading source model:", src_ckpt_path)

student = ResNetFeatureExtractor(
    num_classes=CFG.num_classes,
    in_channels=3
).to(device)

student.load_state_dict(torch.load(src_ckpt_path, map_location=device))
print("✅ Loaded source model")

# Freeze BN
student.features.apply(freeze_bn)

# Start with classifier frozen (will temporarily unfreeze in warmup)
for p in student.classifier.parameters():
    p.requires_grad = False

# =====================================================
# Optimizer
# =====================================================
# We will optimize ALL trainable params (feature extractor + optionally classifier during warmup)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, student.parameters()),
    lr=CFG.sf_lr,
    weight_decay=1e-4
)

# =====================================================
# PM-SFDA Training Loop (UPDATED)
# =====================================================

best_tgt = -1.0
warmup_epochs = 5      # classifier calibration + no consistency early
K_proto = 20           # top-K confident features per class for prototypes
min_proto = 3          # prototype valid if >= min_proto samples

print(f"🚀 Starting PM-SFDA for {CFG.sf_epochs} epochs")

for ep in range(1, CFG.sf_epochs + 1):

    student.train()
    student.features.apply(freeze_bn)

    # ---- Temporary classifier calibration (crucial when conf collapses) ----
    if ep <= warmup_epochs:
        for p in student.classifier.parameters():
            p.requires_grad = True
    else:
        for p in student.classifier.parameters():
            p.requires_grad = False

    # IMPORTANT: re-create optimizer if trainable params changed (warmup unfreezing)
    if ep == 1 or ep == warmup_epochs + 1:
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, student.parameters()),
            lr=CFG.sf_lr,
            weight_decay=1e-4
        )

    # ---- threshold (LOW floor) ----
    th = conf_thresh(ep - 1, CFG.sf_epochs, floor=0.40)
    print(f"\n🔹 Epoch {ep}/{CFG.sf_epochs} | Conf Thresh = {th:.2f}")

    # ---- Debug confidence stats ----
    stats = confidence_stats(student, tgt_unl_loader, max_batches=3)
    if stats is not None:
        cmin, cmean, cmax = stats
        print(f"🔎 Conf stats (min/mean/max): {cmin:.3f} / {cmean:.3f} / {cmax:.3f}")

    # -------------------------------------------------
    # (1) Epoch Prototypes via TOP-K per class (anti-collapse)
    # -------------------------------------------------
    with torch.no_grad():
        feats_by_class = [[] for _ in range(CFG.num_classes)]  # list of (conf, feat)

        for xw, _, _ in tgt_unl_loader:
            xw = xw.to(device)
            logits_w, feats_w = student(xw, return_feat=True)

            probs = torch.softmax(logits_w, dim=1)
            conf, y_hat = probs.max(dim=1)

            feats_w = F.normalize(feats_w, dim=1)

            # collect candidates above threshold
            for i in range(xw.size(0)):
                if conf[i].item() >= th:
                    c = int(y_hat[i].item())
                    feats_by_class[c].append((conf[i].item(), feats_w[i].detach()))

        prototypes = torch.zeros(CFG.num_classes, student.feat_dim, device=device)
        proto_cnt  = torch.zeros(CFG.num_classes, device=device)

        for c in range(CFG.num_classes):
            if len(feats_by_class[c]) == 0:
                continue
            feats_by_class[c].sort(key=lambda t: t[0], reverse=True)
            top_feats = [f for _, f in feats_by_class[c][:K_proto]]
            proto = torch.stack(top_feats, dim=0).mean(dim=0)
            prototypes[c] = F.normalize(proto, dim=0)
            proto_cnt[c] = len(top_feats)

        valid_proto = proto_cnt >= min_proto
        print("🔎 Prototype samples per class (Top-K):", proto_cnt.int().tolist())

    # -------------------------------------------------
    # (2) Adaptation Step
    # -------------------------------------------------
    pbar = tqdm(tgt_unl_loader, desc="PM-SFDA update")

    for xw, xs, _ in pbar:
        xw, xs = xw.to(device), xs.to(device)

        logits_w, feats_w = student(xw, return_feat=True)
        logits_s, feats_s = student(xs, return_feat=True)

        probs = torch.softmax(logits_w, dim=1)
        conf, y_hat = probs.max(dim=1)

        mask = conf >= th
        if mask.sum() == 0:
            continue

        # --- Pseudo-label CE ---
        loss_pl = F.cross_entropy(logits_s[mask], y_hat[mask])

        # --- Prototype attraction (only for classes with valid prototypes) ---
        has_proto = valid_proto[y_hat]
        final_mask = mask & has_proto
        if final_mask.sum() > 0:
            loss_proto = proto_loss(feats_s[final_mask], y_hat[final_mask], prototypes)
        else:
            loss_proto = torch.tensor(0.0, device=device)

        # --- Consistency (off during warmup) ---
        if ep <= warmup_epochs:
            loss_cons = torch.tensor(0.0, device=device)
        else:
            loss_cons = consistency_kl(logits_w, logits_s)

        # --- Total loss ---
        proto_weight = CFG.lambda_proto * min(1.0, ep / 10.0)
        loss = (
            CFG.lambda_pl   * loss_pl +
            proto_weight    * loss_proto +
            CFG.lambda_cons * loss_cons
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in student.parameters() if p.requires_grad], 5.0
        )
        optimizer.step()

        pbar.set_postfix(
            PL=f"{loss_pl.item():.3f}",
            Proto=f"{loss_proto.item():.3f}",
            Cons=f"{loss_cons.item():.3f}",
            Masked=f"{mask.sum().item()}/{xw.size(0)}"
        )

    # -------------------------------------------------
    # (3) Evaluation
    # -------------------------------------------------
    student.eval()
    acc_tgt = accuracy(student, tgt_test_loader)
    print(f"📊 Target Acc = {acc_tgt:.2f}%")

    if acc_tgt > best_tgt:
        best_tgt = acc_tgt
        best_path = os.path.join(CFG.out_dir, "student_PM_SFDA_BEST.pth")
        torch.save(student.state_dict(), best_path)
        print(f"🎉 Saved BEST PM-SFDA model ({best_tgt:.2f}%)")

# =====================================================
# Final Save
# =====================================================
final_path = os.path.join(CFG.out_dir, "student_PM_SFDA_FINAL.pth")
torch.save(student.state_dict(), final_path)
print("✅ PM-SFDA training finished")
print(f"🏁 Best Target Accuracy: {best_tgt:.2f}%")


📁 Loading source model: ./pm_sfda_ckpts_whu_to_eurosat\source_model.pth
✅ Loaded source model
🚀 Starting PM-SFDA for 3 epochs

🔹 Epoch 1/3 | Conf Thresh = 0.90
🔎 Conf stats (min/mean/max): 0.224 / 0.678 / 0.997


KeyboardInterrupt: 

In [12]:
#@title 🔁 PM-SFDA Adaptation (Prototype + Consistency, Conference — FROZEN CLASSIFIER)

import torch
import torch.nn as nn
import torch.nn.functional as F
import os
from tqdm import tqdm

# =====================================================
# Helper Functions
# =====================================================

def freeze_bn(m):
    """Freeze BatchNorm running stats (source-free stability)."""
    if isinstance(m, nn.BatchNorm2d):
        m.eval()

def conf_thresh(epoch, max_epoch, start=CFG.conf_start, end=CFG.conf_end, floor=0.40): #floor=0.40
    """
    Linear schedule with LOW floor (critical for small WHU train sets).
    """
    th = start - (start - end) * (epoch / max_epoch)
    return max(th, floor)

def consistency_kl(logits_w, logits_s, T=CFG.temp_cons):
    """Weak–Strong KL consistency."""
    p_w = torch.softmax(logits_w / T, dim=1).detach()
    log_p_s = torch.log_softmax(logits_s / T, dim=1)
    return F.kl_div(log_p_s, p_w, reduction="batchmean") * (T * T)

def proto_loss(feats, labels, protos):
    """Cosine prototype attraction."""
    feats = F.normalize(feats, dim=1)
    return (1.0 - (feats * protos[labels]).sum(dim=1)).mean()

@torch.no_grad()
def confidence_stats(model, loader, max_batches=3):
    """Diagnostic: confidence distribution on target."""
    model.eval()
    confs = []
    for bi, (xw, _, _) in enumerate(loader):
        if bi >= max_batches:
            break
        xw = xw.to(device)
        logits, _ = model(xw, return_feat=True)
        conf = torch.softmax(logits, dim=1).max(dim=1)[0]
        confs.append(conf.cpu())
    if len(confs) == 0:
        return None
    confs = torch.cat(confs)
    return float(confs.min()), float(confs.mean()), float(confs.max())

# =====================================================
# Load Source Model
# =====================================================
best_tgt = -1.0
warmup_epochs = 5      # classifier calibration + no consistency early
K_proto = 20           # top-K confident features per class for prototypes
min_proto = 3          # prototype valid if >= min_proto samples

src_ckpt_path = os.path.join(CFG.out_dir, "source_model.pth")
print("📁 Loading source model:", src_ckpt_path)

student = ResNetFeatureExtractor(
    num_classes=CFG.num_classes,
    in_channels=3
).to(device)

student.load_state_dict(torch.load(src_ckpt_path, map_location=device))
print("✅ Loaded source model")

# -----------------------------------------------------
# STRICT PM-SFDA REQUIREMENT: FROZEN CLASSIFIER
# -----------------------------------------------------
for p in student.classifier.parameters():
    p.requires_grad = False

# Freeze BatchNorm statistics
student.features.apply(freeze_bn)

# =====================================================
# Optimizer (FEATURE EXTRACTOR ONLY)
# =====================================================
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, student.parameters()),
    lr=CFG.sf_lr,
    weight_decay=1e-4
)

# =====================================================
# PM-SFDA Training Loop
# =====================================================

best_tgt = -1.0
K_proto = 20        # Top-K confident samples per class
min_proto = 3       # Minimum samples to activate prototype

print(f"🚀 Starting PM-SFDA for {CFG.sf_epochs} epochs")

for ep in range(1, CFG.sf_epochs + 1):

    student.train()
    student.features.apply(freeze_bn)

    th = conf_thresh(ep - 1, CFG.sf_epochs)
    print(f"\n🔹 Epoch {ep}/{CFG.sf_epochs} | Conf Thresh = {th:.2f}")

    # ---- Confidence diagnostics ----
    stats = confidence_stats(student, tgt_unl_loader)
    if stats is not None:
        cmin, cmean, cmax = stats
        print(f"🔎 Conf stats (min/mean/max): {cmin:.3f} / {cmean:.3f} / {cmax:.3f}")

    # -------------------------------------------------
    # (1) Epoch-based Prototype Construction (Top-K)
    # -------------------------------------------------
    with torch.no_grad():
        feats_by_class = [[] for _ in range(CFG.num_classes)]

        for xw, _, _ in tgt_unl_loader:
            xw = xw.to(device)
            logits_w, feats_w = student(xw, return_feat=True)

            probs = torch.softmax(logits_w, dim=1)
            conf, y_hat = probs.max(dim=1)
            feats_w = F.normalize(feats_w, dim=1)

            for i in range(xw.size(0)):
                if conf[i].item() >= th:
                    c = int(y_hat[i].item())
                    feats_by_class[c].append((conf[i].item(), feats_w[i]))

        prototypes = torch.zeros(CFG.num_classes, student.feat_dim, device=device)
        proto_cnt  = torch.zeros(CFG.num_classes, device=device)

        for c in range(CFG.num_classes):
            if len(feats_by_class[c]) == 0:
                continue
            feats_by_class[c].sort(key=lambda t: t[0], reverse=True)
            top_feats = [f for _, f in feats_by_class[c][:K_proto]]
            proto = torch.stack(top_feats, dim=0).mean(dim=0)
            prototypes[c] = F.normalize(proto, dim=0)
            proto_cnt[c] = len(top_feats)

        valid_proto = proto_cnt >= min_proto
        print("🔎 Prototype samples per class (Top-K):", proto_cnt.int().tolist())

    # -------------------------------------------------
    # (2) Adaptation Step
    # -------------------------------------------------
    pbar = tqdm(tgt_unl_loader, desc="PM-SFDA update")

    for xw, xs, _ in pbar:
        xw, xs = xw.to(device), xs.to(device)

        logits_w, feats_w = student(xw, return_feat=True)
        logits_s, feats_s = student(xs, return_feat=True)

        probs = torch.softmax(logits_w, dim=1)
        conf, y_hat = probs.max(dim=1)

        mask = conf >= th
        if mask.sum() == 0:
            continue

        # --- Pseudo-label CE ---
        loss_pl = F.cross_entropy(logits_s[mask], y_hat[mask])

        # --- Prototype attraction ---
        has_proto = valid_proto[y_hat]
        final_mask = mask & has_proto
        loss_proto = (
            proto_loss(feats_s[final_mask], y_hat[final_mask], prototypes)
            if final_mask.sum() > 0 else torch.tensor(0.0, device=device)
        )

        # --- Consistency ---
        loss_cons = consistency_kl(logits_w, logits_s)

        # --- Total loss ---
        proto_weight = CFG.lambda_proto * min(1.0, ep / 10.0)
        loss = (
            CFG.lambda_pl   * loss_pl +
            proto_weight    * loss_proto +
            CFG.lambda_cons * loss_cons
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in student.parameters() if p.requires_grad], 5.0
        )
        optimizer.step()

        pbar.set_postfix(
            PL=f"{loss_pl.item():.3f}",
            Proto=f"{loss_proto.item():.3f}",
            Cons=f"{loss_cons.item():.3f}",
            Masked=f"{mask.sum().item()}/{xw.size(0)}"
        )

    # -------------------------------------------------
    # (3) Evaluation
    # -------------------------------------------------
    student.eval()
    acc_tgt = accuracy(student, tgt_test_loader)
    print(f"📊 Target Acc = {acc_tgt:.2f}%")

    if acc_tgt > best_tgt:
        best_tgt = acc_tgt
        best_path = os.path.join(CFG.out_dir, "student_PM_SFDA_BEST.pth")
        torch.save(student.state_dict(), best_path)
        print(f"🎉 Saved BEST PM-SFDA model ({best_tgt:.2f}%)")

# =====================================================
# Final Save
# =====================================================
final_path = os.path.join(CFG.out_dir, "student_PM_SFDA_FINAL.pth")
torch.save(student.state_dict(), final_path)
print("✅ PM-SFDA training finished")
print(f"🏁 Best Target Accuracy: {best_tgt:.2f}%")


📁 Loading source model: ./pm_sfda_ckpts_whu_to_eurosat\source_model.pth
✅ Loaded source model
🚀 Starting PM-SFDA for 3 epochs

🔹 Epoch 1/3 | Conf Thresh = 0.90
🔎 Conf stats (min/mean/max): 0.241 / 0.702 / 0.984
🔎 Prototype samples per class (Top-K): [20, 0, 20, 20, 1, 1]


PM-SFDA update: 100%|██████████| 175/175 [08:51<00:00,  3.04s/it, Cons=0.000, Masked=64/64, PL=0.000, Proto=0.002]


📊 Target Acc = 15.62%
🎉 Saved BEST PM-SFDA model (15.62%)

🔹 Epoch 2/3 | Conf Thresh = 0.83
🔎 Conf stats (min/mean/max): 1.000 / 1.000 / 1.000
🔎 Prototype samples per class (Top-K): [0, 0, 20, 0, 0, 0]


PM-SFDA update: 100%|██████████| 175/175 [10:35<00:00,  3.63s/it, Cons=0.000, Masked=64/64, PL=0.000, Proto=0.000]


📊 Target Acc = 15.62%

🔹 Epoch 3/3 | Conf Thresh = 0.77
🔎 Conf stats (min/mean/max): 1.000 / 1.000 / 1.000
🔎 Prototype samples per class (Top-K): [0, 0, 20, 0, 0, 0]


PM-SFDA update: 100%|██████████| 175/175 [08:33<00:00,  2.93s/it, Cons=0.000, Masked=64/64, PL=0.000, Proto=0.000]


📊 Target Acc = 15.62%
✅ PM-SFDA training finished
🏁 Best Target Accuracy: 15.62%
